# Audio Demo: Moshi Self-Play & Cross-Play

**Two modes:**
- **Self-play:** One model talks to itself (output looped back as input)
- **Cross-play:** Two models have a live duplex conversation (teacher vs student)

**Cross-play output:** 3 audio tracks + 2 text transcripts
1. Model A channel (e.g. teacher speech)
2. Model B channel (e.g. pruned student speech)
3. Dialogue mix (both overlaid)

**VRAM:** Self-play works on T4 (16 GB). Cross-play needs L4 (24 GB) or A100.

## Cell 1: Session Startup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## Cell 2: Configuration

- **self_play:** Model A talks to itself. Works on T4.
- **cross_play:** Model A and Model B have a duplex conversation. Needs L4/A100.

In [ ]:
from pathlib import Path

# ╔══════════════════════════════════════════╗
# ║  MODE                                     ║
# ╚══════════════════════════════════════════╝
PLAY_MODE = "cross_play"  # @param ["self_play", "cross_play"]

# ╔══════════════════════════════════════════╗
# ║  MODEL SELECTION                          ║
# ╚══════════════════════════════════════════╝
MODEL_A = "teacher"  # @param ["teacher", "magnitude_30pct", "wanda_30pct", "sparsegpt_30pct", "v1_scattered_nonuni", "v1_scattered_uni", "v2a_cont_strict_nonuni", "v2a_cont_strict_uni", "v2b_cont_penalized_nonuni", "v2b_cont_penalized_uni", "v2c_cont_relaxed_nonuni", "v2c_cont_relaxed_uni", "v3_collapse_nonuni", "v3_collapse_uni", "magnitude_30pct_L1_kd", "magnitude_30pct_L2_kd", "wanda_30pct_L1_kd", "wanda_30pct_L2_kd", "sparsegpt_30pct_L1_kd", "sparsegpt_30pct_L2_kd", "v1_scattered_nonuni_L1_kd", "v1_scattered_nonuni_L2_kd", "v1_scattered_nonuni_L3_kd", "v1_scattered_nonuni_L4_kd", "v1_scattered_nonuni_L5_kd", "v1_scattered_uni_L2_kd", "v2a_cont_strict_nonuni_L2_kd", "v2a_cont_strict_uni_L2_kd", "v2b_cont_penalized_nonuni_L2_kd", "v2b_cont_penalized_uni_L2_kd", "v2c_cont_relaxed_nonuni_L2_kd", "v2c_cont_relaxed_uni_L2_kd", "v3_collapse_uni_L2_kd", "custom"]

# Model B (only used in cross_play mode)
MODEL_B = "sparsegpt_30pct"  # @param ["teacher", "magnitude_30pct", "wanda_30pct", "sparsegpt_30pct", "v1_scattered_nonuni", "v1_scattered_uni", "v2a_cont_strict_nonuni", "v2a_cont_strict_uni", "v2b_cont_penalized_nonuni", "v2b_cont_penalized_uni", "v2c_cont_relaxed_nonuni", "v2c_cont_relaxed_uni", "v3_collapse_nonuni", "v3_collapse_uni", "magnitude_30pct_L1_kd", "magnitude_30pct_L2_kd", "wanda_30pct_L1_kd", "wanda_30pct_L2_kd", "sparsegpt_30pct_L1_kd", "sparsegpt_30pct_L2_kd", "v1_scattered_nonuni_L1_kd", "v1_scattered_nonuni_L2_kd", "v1_scattered_nonuni_L3_kd", "v1_scattered_nonuni_L4_kd", "v1_scattered_nonuni_L5_kd", "v1_scattered_uni_L2_kd", "v2a_cont_strict_nonuni_L2_kd", "v2a_cont_strict_uni_L2_kd", "v2b_cont_penalized_nonuni_L2_kd", "v2b_cont_penalized_uni_L2_kd", "v2c_cont_relaxed_nonuni_L2_kd", "v2c_cont_relaxed_uni_L2_kd", "v3_collapse_uni_L2_kd", "custom"]

CUSTOM_PATH_A = ""  # @param {type:"string"}
CUSTOM_PATH_B = ""  # @param {type:"string"}

# ╔══════════════════════════════════════════╗
# ║  GENERATION SETTINGS                      ║
# ╚══════════════════════════════════════════╝
DURATION_SECONDS = 30  # @param {type:"slider", min:5, max:300, step:5}
SEED = 42              # @param {type:"integer"}
SEED_TYPE = "noise"    # @param ["noise", "acoustic", "silence"]
TEMPERATURE = 0.8      # @param {type:"number"}
TEMPERATURE_TEXT = 0.7 # @param {type:"number"}
REPETITION_PENALTY = 1.3  # @param {type:"number"}

# ╔══════════════════════════════════════════╗
# ║  PATHS                                    ║
# ╚══════════════════════════════════════════╝
EXPERIMENT_ID = "prune30_v1"
GDRIVE_BASE = "/content/drive/MyDrive/moshilite"
WEIGHTS_DIR = f"{GDRIVE_BASE}/upstream_weights/moshiko"
CHECKPOINT_DIR = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}"
MODELS_DIR = f"{GDRIVE_BASE}/models"
AUDIO_SAMPLES_DIR = f"{GDRIVE_BASE}/audio_samples"

def resolve_model_path(variant, custom_path=""):
    if variant == "teacher":
        return WEIGHTS_DIR, "teacher"
    elif variant == "custom":
        if not custom_path:
            raise ValueError("Set custom path for 'custom' variant")
        return custom_path, "custom"
    elif variant.endswith("_kd"):
        exported = Path(f"{MODELS_DIR}/{variant}.pt")
        if exported.exists():
            return str(exported), "post_kd"
        run_id = variant.removesuffix("_kd")
        best_ckpt = Path(f"{CHECKPOINT_DIR}/kd/{run_id}/checkpoint_best.pt")
        if best_ckpt.exists():
            return str(best_ckpt), "post_kd"
        return str(exported), "post_kd"
    else:
        return f"{CHECKPOINT_DIR}/{variant}.pt", "pruned"

PATH_A, TYPE_A = resolve_model_path(MODEL_A, CUSTOM_PATH_A)
NUM_STEPS = int(DURATION_SECONDS * 12.5)

if PLAY_MODE == "cross_play":
    PATH_B, TYPE_B = resolve_model_path(MODEL_B, CUSTOM_PATH_B)

# ── Print config ──
print(f"{'='*62}")
print(f"  Mode: {PLAY_MODE.upper()}")
print(f"{'='*62}")
print(f"  Model A:    {MODEL_A} ({TYPE_A})")
print(f"  Path A:     {PATH_A}")
if PLAY_MODE == "cross_play":
    print(f"  Model B:    {MODEL_B} ({TYPE_B})")
    print(f"  Path B:     {PATH_B}")
print(f"  Duration:   {DURATION_SECONDS}s ({NUM_STEPS} steps)")
print(f"  Seed:       {SEED} ({SEED_TYPE})")
print(f"  Temp:       {TEMPERATURE} / {TEMPERATURE_TEXT}")
print(f"  Rep. pen.:  {REPETITION_PENALTY}")
print(f"{'='*62}")

# Warn about VRAM
if PLAY_MODE == "cross_play":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 20:
        print(f"\n  WARNING: Cross-play needs ~20 GB VRAM. You have {vram_gb:.0f} GB.")
        print(f"  Consider using L4 or A100 runtime.")


## Cell 3: Load Models + Mimi Codec

Loads Model A (and Model B if cross-play), plus the Mimi audio codec.

In [ ]:
import torch, gc, json
import torch.nn as nn
from pathlib import Path
from moshi.models import loaders
from moshi.models.lm import LMGen

gpu_name = torch.cuda.get_device_name(0).lower()
DTYPE = torch.bfloat16 if any(x in gpu_name for x in ["a100","h100","l4","l40"]) else torch.float16
print(f"Using {DTYPE} on {gpu_name}")

# ── Helper: fix attention metadata after loading pruned weights ──
def _fix_attn_metadata(model):
    """Infer num_heads/embed_dim from actual weight shapes.

    During structured pruning, head_pruning.py updates attn.num_heads and
    attn.embed_dim as plain int attributes. These are NOT saved in state_dict,
    so after loading pruned weights into a fresh model, they retain teacher
    values. This function fixes them by inspecting the weight shapes.
    """
    D_HEAD = 128  # Moshi 7B standard head dimension (4096 / 32)
    for i, layer in enumerate(model.transformer.layers):
        if not hasattr(layer, 'self_attn'):
            continue
        attn = layer.self_attn
        # Infer from out_projs (shape: [d_model, num_heads * d_head])
        if hasattr(attn, 'out_projs') and len(attn.out_projs) > 0:
            out_proj = attn.out_projs[0]
            if hasattr(out_proj, 'weight'):
                actual_embed = out_proj.weight.shape[1]  # in_features
                actual_heads = actual_embed // D_HEAD
                if actual_heads != getattr(attn, 'num_heads', -1):
                    print(f"    Layer {i}: heads {attn.num_heads}->{actual_heads}, embed {getattr(attn, 'embed_dim', '?')}->{actual_embed}")
                    attn.num_heads = actual_heads
                    attn.embed_dim = actual_embed

# ── Helper: load a Moshi LM variant ──
def load_variant(variant_name, model_path, model_type):
    """Load a Moshi LM model for any variant type."""
    weights_path = Path(model_path)

    if variant_name == "teacher":
        config_path = weights_path / "config.json"
        config = None
        moshi_name = "model.safetensors"
        if config_path.exists():
            with open(config_path) as f:
                config = json.load(f)
            moshi_name = config.get("moshi_name", moshi_name)
        lm_path = weights_path / moshi_name
        if not lm_path.exists():
            sf = [f for f in weights_path.glob("*.safetensors")
                  if "tokenizer" not in f.name.lower() and "mimi" not in f.name.lower()]
            if sf: lm_path = sf[0]
            else: raise FileNotFoundError(f"No LM weights in {weights_path}")
        model = loaders.get_moshi_lm(lm_path, lm_kwargs=config, device="cuda", dtype=DTYPE)
        meta = {"pruning_type": "none", "sparsity": 0.0, "kd_trained": False}
        return model, meta

    # Pruned or post-KD
    if not weights_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {weights_path}")
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)
    meta = {
        "pruning_type": ckpt.get("pruning_type", "structured"),
        "pruning_method": ckpt.get("pruning_method", "unknown"),
        "sparsity": ckpt.get("actual_sparsity", ckpt.get("sparsity", 0.0)),
        "kd_trained": model_type == "post_kd",
        "loss_config": ckpt.get("config", {}).get("loss_config", None) if isinstance(ckpt.get("config"), dict) else None,
    }
    sd = ckpt.get("model_state_dict", ckpt)

    # Load base architecture on CPU
    up = Path(WEIGHTS_DIR)
    up_cfg = None
    up_name = "model.safetensors"
    if (up / "config.json").exists():
        with open(up / "config.json") as f:
            up_cfg = json.load(f)
        up_name = up_cfg.get("moshi_name", up_name)
    up_path = up / up_name
    if not up_path.exists():
        sf = [f for f in up.glob("*.safetensors")
              if "tokenizer" not in f.name.lower() and "mimi" not in f.name.lower()]
        if sf: up_path = sf[0]
    model = loaders.get_moshi_lm(up_path, lm_kwargs=up_cfg, device="cpu", dtype=DTYPE)

    # Detect if structured pruning changed layer count
    pruned_layer_idxs = sorted({
        int(k.split(".")[2]) for k in sd
        if k.startswith("transformer.layers.")
    })
    n_pruned_layers = max(pruned_layer_idxs) + 1 if pruned_layer_idxs else len(model.transformer.layers)
    base_n_layers = len(model.transformer.layers)

    if n_pruned_layers < base_n_layers:
        print(f"    Adjusting layers: {base_n_layers} -> {n_pruned_layers}")
        while len(model.transformer.layers) > n_pruned_layers:
            del model.transformer.layers[-1]

    # Force-assign each tensor individually to handle shape mismatches
    # from structured pruning (head/FFN removal changes weight dimensions)
    assigned, skipped = 0, 0
    for name, tensor in sd.items():
        parts = name.split(".")
        try:
            module = model
            for part in parts[:-1]:
                if part.isdigit():
                    module = module[int(part)]
                else:
                    module = getattr(module, part)
            attr_name = parts[-1]
            old = getattr(module, attr_name, None)
            if isinstance(old, nn.Parameter):
                setattr(module, attr_name, nn.Parameter(tensor, requires_grad=False))
            else:
                setattr(module, attr_name, tensor)
            assigned += 1
        except Exception as e:
            skipped += 1
    print(f"    Assigned {assigned} tensors, skipped {skipped}")

    # Fix attention metadata (num_heads, embed_dim) that structured pruning
    # changed but are NOT saved in state_dict (plain int attributes)
    _fix_attn_metadata(model)

    model = model.to("cuda")
    del ckpt, sd
    return model, meta

def model_stats(model, label):
    n_p = sum(p.numel() for p in model.parameters()) / 1e9
    n_nz = sum((p != 0).sum().item() for p in model.parameters()) / 1e9
    print(f"    {label}: {n_p:.3f}B params, {n_nz:.3f}B non-zero")
    return n_p, n_nz

# ── Load Model A ──
print(f"\nLoading Model A: {MODEL_A}")
lm_model_a, meta_a = load_variant(MODEL_A, PATH_A, TYPE_A)
lm_model_a.eval()
n_params_a, n_nonzero_a = model_stats(lm_model_a, "Model A")

# ── Load Model B (cross-play only) ──
if PLAY_MODE == "cross_play":
    print(f"\nLoading Model B: {MODEL_B}")
    lm_model_b, meta_b = load_variant(MODEL_B, PATH_B, TYPE_B)
    lm_model_b.eval()
    n_params_b, n_nonzero_b = model_stats(lm_model_b, "Model B")
else:
    lm_model_b = None
    meta_b = {}
    n_params_b = n_nonzero_b = 0

print(f"\n  VRAM after models: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Load Mimi ──
print(f"\nLoading Mimi codec...")
mimi_candidates = (
    list(Path(WEIGHTS_DIR).glob("tokenizer*.safetensors"))
    + list(Path(WEIGHTS_DIR).glob("mimi*.safetensors"))
)
mimi_path = str(mimi_candidates[0]) if mimi_candidates else None
mimi = loaders.get_mimi(mimi_path, device="cuda")
mimi.eval()
SAMPLE_RATE = mimi.sample_rate
print(f"  Mimi loaded (sample_rate={SAMPLE_RATE})")
print(f"  Total VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Create LMGen wrappers ──
lm_gen_a = LMGen(lm_model_a, use_sampling=True, temp=TEMPERATURE,
                 temp_text=TEMPERATURE_TEXT, top_k=250, top_k_text=25)
if PLAY_MODE == "cross_play":
    lm_gen_b = LMGen(lm_model_b, use_sampling=True, temp=TEMPERATURE,
                     temp_text=TEMPERATURE_TEXT, top_k=250, top_k_text=25)

print(f"\nReady! Mode: {PLAY_MODE}")


## Cell 4: Generate

- **Self-play:** Model A's output is looped back as its own input.
- **Cross-play:** Both models run at every step. Model A hears Model B's
  previous output, and Model B hears Model A's current output. This creates
  a true duplex conversation.

In [ ]:
import time
import numpy as np

dep_q = lm_model_a.dep_q
n_user_cb = lm_model_a.n_q - dep_q
card = lm_model_a.card

# ── Seed tokens ──
rng = torch.Generator(device="cpu").manual_seed(SEED)
if SEED_TYPE == "noise":
    seed_tokens = torch.randint(0, card, (1, n_user_cb, 1), generator=rng).cuda()
elif SEED_TYPE == "acoustic":
    seed_tokens = torch.randint(0, card // 4, (1, n_user_cb, 1), generator=rng).cuda()
elif SEED_TYPE == "silence":
    seed_tokens = torch.zeros(1, n_user_cb, 1, dtype=torch.long).cuda()
else:
    raise ValueError(f"Unknown seed type: {SEED_TYPE}")

# ── Repetition penalty hooks ──
def make_rep_hook(recent_tokens_list):
    def hook(logits):
        if not recent_tokens_list or REPETITION_PENALTY <= 1.0:
            return
        v = logits[0, 0, 0]
        for tok in set(recent_tokens_list):
            if tok < v.shape[0]:
                if v[tok] > 0: v[tok] /= REPETITION_PENALTY
                else:          v[tok] *= REPETITION_PENALTY
    return hook

recent_text_a = []
recent_text_b = []
if REPETITION_PENALTY > 1.0:
    lm_gen_a.on_text_logits_hook = make_rep_hook(recent_text_a)
    if PLAY_MODE == "cross_play":
        lm_gen_b.on_text_logits_hook = make_rep_hook(recent_text_b)

# ── Collect outputs ──
codes_a = []     # Model A audio tokens
codes_b = []     # Model B audio tokens (cross-play only)
text_a = []      # Model A text tokens
text_b = []      # Model B text tokens (cross-play only)

def track_text(tok, recent_list, text_list):
    text_list.append(tok)
    if tok > 3:
        recent_list.append(tok)
        if len(recent_list) > 50:
            recent_list.pop(0)

print(f"Generating {DURATION_SECONDS}s {PLAY_MODE}...")
if PLAY_MODE == "cross_play":
    print(f"  A: {MODEL_A}  <-->  B: {MODEL_B}")
else:
    print(f"  Model: {MODEL_A}")
print(f"  Seed: {SEED} ({SEED_TYPE}) | Steps: {NUM_STEPS}")
print()

t_start = time.time()

try:
    if PLAY_MODE == "self_play":
        # ── Self-play: single model loop ──
        with torch.no_grad(), lm_gen_a.streaming(batch_size=1):
            prev = seed_tokens
            for t in range(NUM_STEPS):
                out = lm_gen_a.step(prev)
                if out is not None:
                    txt = out[0, 0, 0].item()
                    audio = out[0, 1:, 0]
                    codes_a.append(audio.unsqueeze(-1))
                    track_text(txt, recent_text_a, text_a)
                    prev = audio.unsqueeze(0).unsqueeze(-1).long()
                if (t+1) % 62 == 0:
                    e = time.time() - t_start
                    print(f"   [{len(codes_a)/12.5:.0f}s / {DURATION_SECONDS}s] {e:.1f}s elapsed")

    else:
        # ── Cross-play: two models talking to each other ──
        with torch.no_grad(), lm_gen_a.streaming(batch_size=1), lm_gen_b.streaming(batch_size=1):
            prev_b_output = seed_tokens  # Initial input to Model A

            for t in range(NUM_STEPS):
                # Model A: receives Model B's previous output
                out_a = lm_gen_a.step(prev_b_output)

                if out_a is not None:
                    txt_a = out_a[0, 0, 0].item()
                    audio_a = out_a[0, 1:, 0]
                    codes_a.append(audio_a.unsqueeze(-1))
                    track_text(txt_a, recent_text_a, text_a)
                    input_for_b = audio_a.unsqueeze(0).unsqueeze(-1).long()
                else:
                    input_for_b = seed_tokens

                # Model B: receives Model A's current output
                out_b = lm_gen_b.step(input_for_b)

                if out_b is not None:
                    txt_b = out_b[0, 0, 0].item()
                    audio_b = out_b[0, 1:, 0]
                    codes_b.append(audio_b.unsqueeze(-1))
                    track_text(txt_b, recent_text_b, text_b)
                    prev_b_output = audio_b.unsqueeze(0).unsqueeze(-1).long()
                else:
                    prev_b_output = seed_tokens

                if (t+1) % 62 == 0:
                    e = time.time() - t_start
                    n = max(len(codes_a), len(codes_b))
                    print(f"   [{n/12.5:.0f}s / {DURATION_SECONDS}s] {e:.1f}s elapsed, "
                          f"{e/(t+1)*1000:.0f} ms/step")

finally:
    lm_gen_a.on_text_logits_hook = None
    if PLAY_MODE == "cross_play":
        lm_gen_b.on_text_logits_hook = None

t_elapsed = time.time() - t_start

# ── Stack results ──
if not codes_a:
    raise RuntimeError("No output from Model A.")

moshi_codes_a = torch.cat(codes_a, dim=-1).unsqueeze(0)  # [1, 8, T]
text_arr_a = np.array(text_a, dtype=np.int32)

if PLAY_MODE == "cross_play" and codes_b:
    moshi_codes_b = torch.cat(codes_b, dim=-1).unsqueeze(0)
    text_arr_b = np.array(text_b, dtype=np.int32)
else:
    moshi_codes_b = None
    text_arr_b = np.array([], dtype=np.int32)

actual_duration = moshi_codes_a.shape[-1] / 12.5
ms_per_step = (t_elapsed / NUM_STEPS) * 1000
rtf = actual_duration / t_elapsed if t_elapsed > 0 else 0

print(f"\nGeneration complete!")
print(f"   Duration:  {actual_duration:.1f}s")
print(f"   Wall time: {t_elapsed:.1f}s")
print(f"   Speed:     {ms_per_step:.0f} ms/step ({rtf:.2f}x RT)")
if PLAY_MODE == "cross_play":
    print(f"   A tokens:  {moshi_codes_a.shape[-1]}")
    print(f"   B tokens:  {moshi_codes_b.shape[-1] if moshi_codes_b is not None else 0}")

## Cell 5: Decode & Play Audio

Moves LM models to CPU, decodes audio tokens with Mimi, and plays inline.

**Cross-play:** 3 audio players (A, B, mixed dialogue). **Self-play:** 1 player.

In [ ]:
import gc
import IPython.display as ipd
from IPython.display import HTML, display

# ── Offload LM models to free VRAM ──
print("Offloading LM models to CPU...")
codes_a_cpu = moshi_codes_a.cpu()
codes_b_cpu = moshi_codes_b.cpu() if moshi_codes_b is not None else None

lm_model_a.cpu()
if PLAY_MODE == "cross_play" and lm_model_b is not None:
    lm_model_b.cpu()
try: del lm_gen_a
except: pass
try: del lm_gen_b
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"  VRAM after offload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Chunked decode ──
CHUNK_TOKENS = 125

def decode_chunked(codec, codes_cpu, chunk_size=CHUNK_TOKENS):
    total_t = codes_cpu.shape[-1]
    wav_chunks = []
    for start in range(0, total_t, chunk_size):
        end = min(start + chunk_size, total_t)
        chunk = codes_cpu[:, :, start:end].cuda()
        with torch.no_grad():
            wav_chunk = codec.decode(chunk)
        wav_chunks.append(wav_chunk.cpu())
        del chunk
        torch.cuda.empty_cache()
    return torch.cat(wav_chunks, dim=-1)

def normalize_audio(audio):
    peak = max(abs(audio.max()), abs(audio.min()), 1e-8)
    return audio / peak * 0.95 if peak > 1.0 else audio

# ── Decode Model A ──
print("Decoding Model A audio...")
wav_a = decode_chunked(mimi, codes_a_cpu)
audio_a = normalize_audio(wav_a[0, 0].numpy())

# ── Decode Model B (cross-play) ──
if codes_b_cpu is not None:
    print("Decoding Model B audio...")
    wav_b = decode_chunked(mimi, codes_b_cpu)
    audio_b = normalize_audio(wav_b[0, 0].numpy())
    # Mix dialogue
    min_len = min(len(audio_a), len(audio_b))
    audio_mix = normalize_audio(0.5 * audio_a[:min_len] + 0.5 * audio_b[:min_len])
else:
    audio_b = None
    audio_mix = None

# ── Decode text ──
def decode_text(text_arr):
    try:
        from sentencepiece import SentencePieceProcessor
        tok_path = Path(WEIGHTS_DIR) / "tokenizer_spm_32k_3.model"
        if tok_path.exists() and len(text_arr) > 0:
            sp = SentencePieceProcessor(str(tok_path))
            meaningful = [int(t) for t in text_arr if t > 3]
            if meaningful:
                return sp.decode(meaningful)
    except: pass
    return ""

transcript_a = decode_text(text_arr_a)
transcript_b = decode_text(text_arr_b) if len(text_arr_b) > 0 else ""

meaningful_a = sum(1 for t in text_arr_a if t > 3)
meaningful_b = sum(1 for t in text_arr_b if t > 3) if len(text_arr_b) > 0 else 0

# ── Display ──
if PLAY_MODE == "cross_play":
    display(HTML(f"""
    <div style="background: #1a1a2e; color: #eee; padding: 20px; border-radius: 12px; margin: 10px 0; font-family: 'Segoe UI', sans-serif;">
      <h2 style="color: #e94560; margin-top: 0;">Cross-Play Dialogue</h2>
      <table style="width: 100%; color: #ccc; font-size: 14px;">
        <tr><td style="padding: 2px 10px;">Model A</td><td>{MODEL_A} ({TYPE_A}) | {n_params_a:.3f}B params, {n_nonzero_a:.3f}B non-zero</td></tr>
        <tr><td style="padding: 2px 10px;">Model B</td><td>{MODEL_B} ({TYPE_B}) | {n_params_b:.3f}B params, {n_nonzero_b:.3f}B non-zero</td></tr>
        <tr><td style="padding: 2px 10px;">Duration</td><td>{actual_duration:.1f}s | Seed {SEED} ({SEED_TYPE})</td></tr>
        <tr><td style="padding: 2px 10px;">Speed</td><td>{ms_per_step:.0f} ms/step ({rtf:.2f}x real-time)</td></tr>
      </table>
    </div>
    """))

    # Transcripts
    if transcript_a:
        display(HTML(f"""
        <div style="background: #16213e; color: #a8d8ea; padding: 12px; border-radius: 8px; margin: 8px 0; font-family: monospace; font-size: 12px; max-height: 120px; overflow-y: auto;">
          <b>Model A inner monologue ({meaningful_a} tokens):</b><br/>
          {transcript_a[:400]}{'...' if len(transcript_a) > 400 else ''}
        </div>"""))
    if transcript_b:
        display(HTML(f"""
        <div style="background: #1a2e16; color: #a8eab0; padding: 12px; border-radius: 8px; margin: 8px 0; font-family: monospace; font-size: 12px; max-height: 120px; overflow-y: auto;">
          <b>Model B inner monologue ({meaningful_b} tokens):</b><br/>
          {transcript_b[:400]}{'...' if len(transcript_b) > 400 else ''}
        </div>"""))

    # 3 audio players
    display(HTML(f'<h3 style="color: #e94560;">Model A: {MODEL_A}</h3>'))
    display(ipd.Audio(audio_a, rate=SAMPLE_RATE))

    display(HTML(f'<h3 style="color: #48bb78;">Model B: {MODEL_B}</h3>'))
    display(ipd.Audio(audio_b, rate=SAMPLE_RATE))

    display(HTML('<h3 style="color: #805ad5;">Full Dialogue (A + B mixed)</h3>'))
    display(ipd.Audio(audio_mix, rate=SAMPLE_RATE))

else:
    # Self-play: single player
    display(HTML(f"""
    <div style="background: #1a1a2e; color: #eee; padding: 20px; border-radius: 12px; margin: 10px 0; font-family: 'Segoe UI', sans-serif;">
      <h2 style="color: #e94560; margin-top: 0;">Self-Play: {MODEL_A}</h2>
      <table style="width: 100%; color: #ccc; font-size: 14px;">
        <tr><td style="padding: 2px 10px;">Model</td><td>{MODEL_A} ({TYPE_A}) | {n_params_a:.3f}B params</td></tr>
        <tr><td style="padding: 2px 10px;">Duration</td><td>{actual_duration:.1f}s | Seed {SEED}</td></tr>
        <tr><td style="padding: 2px 10px;">Speed</td><td>{ms_per_step:.0f} ms/step ({rtf:.2f}x RT)</td></tr>
      </table>
    </div>
    """))
    if transcript_a:
        display(HTML(f"""
        <div style="background: #16213e; color: #a8d8ea; padding: 12px; border-radius: 8px; margin: 8px 0; font-family: monospace; font-size: 12px; max-height: 120px; overflow-y: auto;">
          <b>Inner monologue ({meaningful_a} tokens):</b><br/>{transcript_a[:500]}
        </div>"""))
    display(HTML(f'<h3 style="color: #e94560;">Self-Play Audio</h3>'))
    display(ipd.Audio(audio_a, rate=SAMPLE_RATE))

print(f"\nNote: LM models moved to CPU. Re-run Cell 3 to regenerate.")

## Cell 6: Save to Google Drive

**Cross-play output:**
```
audio_samples/cross_{A}_vs_{B}_{seed}_{ts}/
  ├── model_a.wav, model_b.wav, dialogue.wav
  └── metadata.json
```
**Self-play output:**
```
audio_samples/{variant}_{seed}_{ts}/
  ├── self_play.wav
  └── metadata.json
```

In [ ]:
import soundfile as sf
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

if PLAY_MODE == "cross_play":
    folder = f"cross_{MODEL_A}_vs_{MODEL_B}_{SEED}_{ts}"
else:
    folder = f"{MODEL_A}_{SEED}_{ts}"

save_dir = Path(AUDIO_SAMPLES_DIR) / folder
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving to: {save_dir}")

if PLAY_MODE == "cross_play":
    sf.write(str(save_dir / "model_a.wav"), audio_a, SAMPLE_RATE)
    sf.write(str(save_dir / "model_b.wav"), audio_b, SAMPLE_RATE)
    sf.write(str(save_dir / "dialogue.wav"), audio_mix, SAMPLE_RATE)
else:
    sf.write(str(save_dir / "self_play.wav"), audio_a, SAMPLE_RATE)

# ── Metadata ──
metadata = {
    "play_mode": PLAY_MODE,
    # Model A
    "model_a": MODEL_A,
    "model_a_type": TYPE_A,
    "model_a_path": PATH_A,
    "model_a_pruning_type": meta_a.get("pruning_type", "none"),
    "model_a_pruning_method": meta_a.get("pruning_method", "none"),
    "model_a_sparsity": meta_a.get("sparsity", 0.0),
    "model_a_kd_trained": meta_a.get("kd_trained", False),
    "model_a_loss_config": meta_a.get("loss_config", None),
    "model_a_params_B": round(n_params_a, 3),
    "model_a_nonzero_B": round(n_nonzero_a, 3),
    "model_a_text_transcript": transcript_a[:2000] if transcript_a else "",
    "model_a_text_tokens_meaningful": meaningful_a,
}

if PLAY_MODE == "cross_play":
    metadata.update({
        "model_b": MODEL_B,
        "model_b_type": TYPE_B,
        "model_b_path": PATH_B,
        "model_b_pruning_type": meta_b.get("pruning_type", "none"),
        "model_b_pruning_method": meta_b.get("pruning_method", "none"),
        "model_b_sparsity": meta_b.get("sparsity", 0.0),
        "model_b_kd_trained": meta_b.get("kd_trained", False),
        "model_b_loss_config": meta_b.get("loss_config", None),
        "model_b_params_B": round(n_params_b, 3),
        "model_b_nonzero_B": round(n_nonzero_b, 3),
        "model_b_text_transcript": transcript_b[:2000] if transcript_b else "",
        "model_b_text_tokens_meaningful": meaningful_b,
    })

metadata.update({
    "experiment_id": EXPERIMENT_ID,
    "seed": SEED,
    "seed_type": SEED_TYPE,
    "duration_seconds": round(actual_duration, 1),
    "num_steps": NUM_STEPS,
    "temperature": TEMPERATURE,
    "temperature_text": TEMPERATURE_TEXT,
    "repetition_penalty": REPETITION_PENALTY,
    "generation_speed_ms_per_step": round(ms_per_step, 1),
    "real_time_factor": round(rtf, 2),
    "generation_wall_time_s": round(t_elapsed, 1),
    "gpu": torch.cuda.get_device_name(0),
    "sample_rate": SAMPLE_RATE,
    "timestamp": datetime.now().isoformat(),
})

with open(save_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

total_mb = sum(f.stat().st_size for f in save_dir.iterdir()) / 1e6

print(f"\nSaved!")
for f in sorted(save_dir.iterdir()):
    print(f"   {f.name}")
print(f"   Total: {total_mb:.1f} MB")